In [5]:
# Jupyter notebook display configuration
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%gui qt

# Data analysis libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Image processing libraries
from skimage.measure import label, regionprops
from skimage.io import imread
from tifffile import imread

# Image visualization
import napari

# File and path handling
import os
from glob import glob
from PIL import Image

# Colony Tracking with Frame Persistence Filtering for Replicate 1

## Overview
This notebook tracks colonies across multiple time-lapse imaging positions, assigns globally unique colony IDs, handles merged colonies, and filters out short-lived detections.

## Input Data
- 3D binary masks (`colonies_mask.npy`) from multiple imaging positions
- Each mask has dimensions: (time, y, x)

## Tracking Algorithm
1. **Frame-by-Frame Labeling**: Colonies are labeled using `skimage.measure.label`
2. **ID Propagation**: Colony IDs are assigned based on spatial overlap between consecutive frames
3. **Merge Detection**: When multiple colonies merge, they receive IDs offset by `MERGED_ID_OFFSET`
4. **Global Uniqueness**: Colony IDs are globally unique across all positions using shared counters

## Filtering Criteria
- **Frame Persistence**: Colonies must appear in more than `FILTER` frames (default: 3)
- **Growth Filter**: Colonies must grow by at least a factor of `GROWTH_THRESHOLD`
- **Manual Curation**: Artifact colonies can be manually excluded

## Output DataFrames
- `tracking_df`: Complete tracking data across all frames and positions
- `first_colonies_df`: Metadata for the first appearance of each colony
- `non_merged_df`: Filtered summary of non-merged colonies only

## Summary
The analysis prints the number of non-merged, persistent colonies detected per imaging position.

In [6]:
# Initialize global tracking containers
all_tracking_data = []
all_first_colonies_data = []

# Base directory for positions (relative path)
base_dir = '../../../../Image_Data/Figure1/1E_wt/replicate_1'
positions = ['pos0', 'pos2', 'pos3', 'pos6', 'pos7', 'pos8', 'pos9', 'pos11', 'pos12', 'pos14']

# Configuration parameters
MERGED_ID_OFFSET = 1000000  # Offset to distinguish merged colonies
FILTER = 3  # Minimum number of frames a colony must appear in

# Global tracking dictionaries
colony_frame_counts = {}  # Track frame count for each colony

# Global counters (shared across all positions for unique IDs)
global_colony_id_counter = 1
global_merged_colony_counter = 1

# Process each position
for pos in positions:
    colonies_mask_path = os.path.join(base_dir, pos, 'colonies_mask.npy')
    colonies_mask = np.load(colonies_mask_path)

    # Position-specific tracking variables
    tracking_data = []
    merged_colony_map = {}
    first_colonies_appearance = set()
    first_colonies_appearance_data = []
    previous_colony_labels = None
    previous_colony_ids = None

    # Process each frame in the time-lapse
    for frame_idx in range(len(colonies_mask)):
        current_colony_labels, num_colonies = label(colonies_mask[frame_idx], return_num=True)
        current_colony_ids = {}

        if previous_colony_labels is not None:
            # Track colonies by overlap with previous frame
            for current_label in range(1, num_colonies + 1):
                current_mask = current_colony_labels == current_label
                overlap = previous_colony_labels[current_mask]
                overlapping_colonies = np.unique(overlap[overlap > 0])

                if len(overlapping_colonies) > 1:
                    # Handle merged colonies
                    previous_ids = np.array([previous_colony_ids[label] for label in overlapping_colonies])
                    if np.any(previous_ids >= MERGED_ID_OFFSET):
                        # Reuse existing merged ID
                        merged_id = np.min(previous_ids[previous_ids >= MERGED_ID_OFFSET])
                    else:
                        # Create new merged ID
                        merged_id = MERGED_ID_OFFSET + global_merged_colony_counter
                        global_merged_colony_counter += 1
                    current_colony_ids[current_label] = merged_id
                    merged_colony_map[current_label] = merged_id
                    
                elif len(overlapping_colonies) == 1:
                    # Continue tracking existing colony
                    overlapping_label = overlapping_colonies[0]
                    current_colony_ids[current_label] = previous_colony_ids[overlapping_label]
                else:
                    # New colony appeared
                    current_colony_ids[current_label] = global_colony_id_counter
                    global_colony_id_counter += 1
        else:
            # Initialize IDs for first frame
            for current_label in range(1, num_colonies + 1):
                current_colony_ids[current_label] = global_colony_id_counter
                global_colony_id_counter += 1

        # Extract region properties and track data
        for region in regionprops(current_colony_labels):
            colony_id = current_colony_ids[region.label]

            # Update frame count for this colony
            colony_frame_counts[colony_id] = colony_frame_counts.get(colony_id, 0) + 1

            # Record first appearance
            if colony_id not in first_colonies_appearance:
                first_colonies_appearance.add(colony_id)
                first_colonies_appearance_data.append({
                    "position": pos,
                    "frame": frame_idx,
                    "colony_id": colony_id,
                    "area": region.area,
                    "centroid": region.centroid,
                    "bbox": region.bbox,
                    "replicate": "replicate1"
                })

            # Record all tracking data
            tracking_data.append({
                "position": pos,
                "frame": frame_idx,
                "colony_id": colony_id,
                "area": region.area,
                "centroid": region.centroid,
                "bbox": region.bbox,
                "replicate": "replicate1"
            })

        # Update for next iteration
        previous_colony_labels = current_colony_labels
        previous_colony_ids = current_colony_ids

    # Add position data to global containers
    all_tracking_data.extend(tracking_data)
    all_first_colonies_data.extend(first_colonies_appearance_data)

# Convert to DataFrames
tracking_df = pd.DataFrame(all_tracking_data)
first_colonies_df = pd.DataFrame(all_first_colonies_data)

# Apply frame count filter
valid_colonies = [col_id for col_id, count in colony_frame_counts.items() if count > FILTER]
tracking_df = tracking_df[tracking_df['colony_id'].isin(valid_colonies)]
first_colonies_df = first_colonies_df[first_colonies_df['colony_id'].isin(valid_colonies)]

# Filter non-merged colonies
non_merged_df = first_colonies_df[first_colonies_df['colony_id'] < MERGED_ID_OFFSET]

# Print summary
print("Non-merged colony counts per position (after frame persistence filter):")
print(non_merged_df.groupby('position')['colony_id'].nunique())

Non-merged colony counts per position (after frame persistence filter):
position
pos0     38
pos11    32
pos12    82
pos14    45
pos2     94
pos3     66
pos6     57
pos7     28
pos8     75
pos9     57
Name: colony_id, dtype: int64


# Colony Growth Filtering

## Objective
Filter tracked colonies based on their growth from first to last appearance, retaining only those that meet a minimum growth criterion.

## Parameters
- `GROWTH_THRESHOLD`: Minimum required growth factor (default: 1.3, representing 30% increase in area)

## Algorithm
1. **Compute Growth Factor**:
   - Sort tracking data by colony ID and frame number
   - Calculate ratio of final area to initial area for each colony
   
2. **Select Valid Colonies**:
   - Retain only colonies with growth factor ≥ `GROWTH_THRESHOLD`
   
3. **Filter DataFrames**:
   - Update both `tracking_df` and `first_colonies_df` to include only valid colonies
   
4. **Exclude Merged Colonies**:
   - Filter out colonies with IDs above `MERGED_ID_OFFSET`
   
5. **Output Summary**:
   - Print the number of non-merged, growing colonies per position

## Rationale
This filter removes spurious detections and ensures that only genuine, growing colonies are analyzed downstream.

In [7]:
# Growth threshold parameter
GROWTH_THRESHOLD = 1.3

# Compute growth factor from first to last frame per colony
tracking_df_sorted = tracking_df.sort_values(by=['colony_id', 'frame'])
colony_endpoints = tracking_df_sorted.groupby('colony_id')['area'].agg(['first', 'last'])
colony_endpoints['growth_factor'] = colony_endpoints['last'] / colony_endpoints['first']

# Identify colonies that meet the growth threshold
valid_colonies = set(colony_endpoints[colony_endpoints['growth_factor'] >= GROWTH_THRESHOLD].index)

# Filter tracking_df and first_colonies_df to include only valid colonies
tracking_df = tracking_df[tracking_df['colony_id'].isin(valid_colonies)].copy()
first_colonies_df = first_colonies_df[first_colonies_df['colony_id'].isin(valid_colonies)].copy()

# Filter to non-merged colonies only
non_merged_df = first_colonies_df[first_colonies_df['colony_id'] < MERGED_ID_OFFSET]

# Print summary
print(f"\nColonies passing growth filter (≥{GROWTH_THRESHOLD}x growth):")
print(non_merged_df.groupby('position')['colony_id'].nunique())


Colonies passing growth filter (≥1.3x growth):
position
pos0     15
pos11    12
pos12    27
pos14    19
pos2     28
pos3     20
pos6     19
pos7     10
pos8     22
pos9     15
Name: colony_id, dtype: int64


# Export Colony Metadata to CSV

## Purpose
Prepare and export a CSV file containing metadata for non-merged colonies at their first frame of appearance.

## Processing Steps
1. **Exclude Merged Colonies**:
   - Filter `first_colonies_df` to include only colonies with IDs below `MERGED_ID_OFFSET`

2. **Format Centroid Coordinates**:
   - Extract `x` and `y` from centroid tuples
   - Note: skimage centroids are in (row, col) format, corresponding to (y, x)

3. **Create Export DataFrame**:
   - Include columns: `colony_id`, `x`, `y`, `frame`, `position`, `replicate`
   - Ensures proper formatting for downstream analysis

4. **Save to CSV**:
   - Write formatted DataFrame to specified output path
   - Use relative path for portability

## Output
- CSV file with one row per colony
- Contains spatial coordinates and temporal information for first detection

In [ ]:
# Exclude merged colonies (IDs ≥ MERGED_ID_OFFSET)
colonies_to_export = first_colonies_df[
    first_colonies_df['colony_id'] < MERGED_ID_OFFSET
].copy()

# Create export DataFrame with formatted coordinates
# Note: skimage centroid is (row, col) which corresponds to (y, x)
final_export_df = pd.DataFrame({
    'colony_id': colonies_to_export['colony_id'],
    'x': colonies_to_export['centroid'].apply(lambda c: c[1] if isinstance(c, tuple) else np.nan),
    'y': colonies_to_export['centroid'].apply(lambda c: c[0] if isinstance(c, tuple) else np.nan),
    'frame': colonies_to_export['frame'],
    'position': colonies_to_export['position'],
    'replicate': "replicate1"
})

# Define output path (relative) and save
OUTPUT_PATH = '2_first_colonies_wt_rep1.csv'
final_export_df.to_csv(OUTPUT_PATH, index=False)

# Print confirmation
print(f"\n{'='*50}")
print(f"Export Summary:")
print(f"  Total colonies exported: {len(final_export_df)}")
print(f"  Output file: {OUTPUT_PATH}")
print(f"{'='*50}")

# Interactive Visualization in Napari

## Purpose
Visualize first-appearance colonies overlaid on the time-lapse mCherry image stack for quality control and validation.

## Workflow
1. **Load Colony Metadata**: Read CSV with colony centroids, frames, and IDs
2. **Load Image Stack**: Stack PNG images from selected position into 3D array (time, y, x)
3. **Filter Colonies**: Select colonies for the specified position
4. **Format Coordinates**: Convert to (frame, y, x) format for Napari points layer
5. **Display**: 
   - Image stack with magenta colormap
   - Colony centroids as yellow points
   - Colony IDs as text labels

## Usage
This enables interactive inspection of tracking results, validation of colony detection, and exploration of temporal dynamics.

In [ ]:
# Load colony metadata (relative path)
csv_path = '2_first_colonies_wt_rep1.csv'
first_colonies_df = pd.read_csv(csv_path)

# Define directory containing PNG frames (relative path)
png_dir = "../../../../Image_Data/Figure1/1E_wt/replicate_1/pos7/mcherry/cut_im"

# Get sorted list of PNG files
png_files = sorted(glob(os.path.join(png_dir, "*_pos7_mcherry_frame*_cut.png")))

# Load all PNGs into a 3D NumPy array (T, Y, X)
mcherry_stack = np.array([np.array(Image.open(f)) for f in png_files])

# Filter colonies for selected position
pos = 'pos7'
subset = first_colonies_df[first_colonies_df['position'] == pos]

# Convert centroid coordinates to (t, y, x) format for Napari points layer
points = np.array([
    [row['frame'], row['y'], row['x']] for _, row in subset.iterrows()
])

# Prepare colony IDs as labels
properties = {'colony_id': subset['colony_id'].tolist()}

# Launch Napari viewer
viewer = napari.Viewer()
viewer.add_image(mcherry_stack, name='mCherry Time-Lapse', colormap='magenta')
viewer.add_points(
    points,
    name='First Colonies',
    properties=properties,
    face_color='yellow',
    size=6,
    symbol='o',
    text='colony_id'
)

print(f"Displaying {len(points)} colonies for {pos}")

# Manual Curation of Colony Data

## Purpose
Remove manually identified artifact colonies or false detections from the exported dataset.

## Method
- Defines a dictionary of colony IDs to exclude, organized by position
- Filters the CSV to remove specified colonies
- Saves the cleaned dataset back to the same file

## Rationale
Despite automated filtering, some artifacts may persist and require manual removal for accurate downstream analysis.

In [ ]:
# Define file path (relative)
FILE_PATH = '2_first_colonies_wt_rep1.csv'

# Load the CSV
df = pd.read_csv(FILE_PATH)
initial_count = len(df)

# Define colonies to remove per position (manually curated)
colonies_to_remove = {
    'pos0': [165, 317, 539, 904],
    'pos2': [1229, 1332, 1331, 2005, 2011, 2021, 2272],
    'pos3': [2829, 3032, 3169, 3355, 3477, 3685],
    'pos6': [4745, 4981, 5207],
    'pos7': [5469],
    'pos8': [7158, 7284, 7311, 7725, 7598, 7769],
    'pos9': [9175],
    'pos11': [9583, 9671, 10322],
    'pos12': [10520, 10571, 10944, 11111, 11143, 11482],
    'pos14': []
}

# Apply filtering
for pos, colony_ids in colonies_to_remove.items():
    df = df[~((df['position'] == pos) & (df['colony_id'].isin(colony_ids)))]

# Save the filtered DataFrame
df.to_csv(FILE_PATH, index=False)

# Print confirmation
final_count = len(df)
removed_count = initial_count - final_count
print(f"\nManual Curation Complete:")
print(f"  Initial colonies: {initial_count}")
print(f"  Removed colonies: {removed_count}")
print(f"  Final colonies: {final_count}")
print(f"  Updated file: {FILE_PATH}")

# Spatial and Temporal Visualization of Colony Emergence

## Purpose
Visualize the spatial distribution and temporal appearance of first colonies across all imaging positions.

## Visualization Features
- **Separate subplot for each position**
- **Spatial coordinates**: Colony centroids in micrometers (scaled by 0.065 µm/pixel)
- **Color coding**: Points colored by appearance time using 'viridis' colormap
- **Labels**: Each point labeled with its colony ID
- **Coordinate system**: Y-axis inverted to match image orientation

## Output
Multi-panel figure showing spatial and temporal patterns of colony emergence, useful for assessing:
- Spatial heterogeneity across the imaging field
- Temporal dynamics of colony appearance
- Position-to-position variability

In [ ]:
# Load the exported DataFrame (relative path)
csv_path = '2_first_colonies_wt_rep1.csv'
first_colonies_df = pd.read_csv(csv_path)

# Calculate appearance time in hours
first_colonies_df['hours'] = first_colonies_df['frame'] * (5 / 60)  # 5 min/frame

# Get unique positions
positions = sorted(first_colonies_df['position'].unique())

# Create subplots
fig, axes = plt.subplots(len(positions), 1, figsize=(6, 5 * len(positions)))

# Handle single position case
if len(positions) == 1:
    axes = [axes]

# Plot each position
for ax, pos in zip(axes, positions):
    subset = first_colonies_df[first_colonies_df['position'] == pos]
    
    # Convert pixel coordinates to micrometers
    x_um = subset['x'] * 0.065
    y_um = subset['y'] * 0.065
    hours = subset['hours']

    # Scatter plot with time-based coloring
    sc = ax.scatter(x_um, y_um, c=hours, cmap='viridis', s=100, alpha=0.8, edgecolors='black', linewidth=0.5)
    
    # Add colony ID labels
    for _, row in subset.iterrows():
        ax.text(row['x'] * 0.065, row['y'] * 0.065, str(row['colony_id']), 
                fontsize=8, ha='center', va='center', weight='bold')

    # Formatting
    ax.set_title(f'Colony Emergence at {pos} (n={len(subset)})', fontsize=14, weight='bold')
    ax.set_xlabel('x (µm)', fontsize=12)
    ax.set_ylabel('y (µm)', fontsize=12)
    ax.set_xlim(0, 50)
    ax.set_ylim(0, 50)
    ax.invert_yaxis()  # Match image coordinates
    ax.grid(False)
    ax.set_aspect('equal')

    # Add colorbar
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Appearance Time (hours)', fontsize=11)

plt.tight_layout()
plt.show()